# CS5811 — Phase 3: Feature Engineering

**Phase 3 scope:** 1) one-hot encode categoricals  2) engineer target bins (Low/Medium/High)  3) form derived feature matrix  4) export processed dataset for Phase 4.

Scaling itself happens in Phase 4 (after the train/test split, fit on train only) so the test set stays truly held-out. This phase only encodes and bins data — it doesn't touch the train/test boundary.

## Setup

In [ ]:
import os, joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

os.makedirs('Output', exist_ok=True)

In [ ]:
df = pd.read_csv('Output/cleaned_dataset.csv')
print(df.shape)
df.head()

---
# Phase 3 — Feature Engineering

### 1 · One-hot encode categorical features

In [ ]:
df_enc = pd.get_dummies(df, columns=['gender', 'occupation'], drop_first=True)
df_enc.shape

`drop_first=True` avoids the dummy-variable trap for any linear models in the pipeline (Logistic Regression in Phase 5).

### 2 · Engineer target bins

In [ ]:
bins, labels = [-0.1, 3, 6, 10], ['Low', 'Medium', 'High']

df_enc['stress_class'] = pd.cut(df_enc['stress_level'],
                                bins=bins, labels=labels)
df_enc['sleep_class']  = pd.cut(df_enc['sleep_quality_score'],
                                bins=bins, labels=labels)

print(df_enc['stress_class'].value_counts(), '\n')
print(df_enc['sleep_class'].value_counts())

Same binning scheme for both targets. Classes are imbalanced — Phase 5 onwards uses `class_weight='balanced'` and macro-F1 to handle it.

### 3 · Form derived feature matrix

In [ ]:
target_cols  = ['stress_level', 'sleep_quality_score', 'stress_class', 'sleep_class']
feature_cols = [c for c in df_enc.columns if c not in target_cols]

X = df_enc[feature_cols].astype(float)

print('feature matrix:', X.shape)
print('targets        :', target_cols)

### 4 · Fit label encoder for class targets

In [ ]:
label_enc = LabelEncoder().fit(['Low', 'Medium', 'High'])
print('classes:', label_enc.classes_)
print('mapping:', dict(zip(label_enc.classes_, label_enc.transform(label_enc.classes_))))

Consistent string → integer mapping for classifiers that need numeric targets (e.g. PyTorch MLP).

### 5 · Export processed dataset

In [ ]:
df_enc.to_csv('Output/processed_data.csv', index=False)

joblib.dump(label_enc, 'Output/label_encoder.pkl')
pd.Series(list(X.columns)).to_csv('Output/feature_names.csv',
                                  index=False, header=['feature'])

print('Saved:')
for f in sorted(os.listdir('Output')):
    print(' -', f)



Next: **Phase 4 — Train-Test Split & Scaling** (the actual `train_test_split` + StandardScaler fit on train only).